In [93]:
import sys,os

fpath = os.path.dirname(os.path.abspath("C:/Users/au646858/OneDrive - Aarhus Universitet/Documents/_repositories/sea-anemone/app/session.py"))
if fpath not in sys.path:
    sys.path.append(fpath)

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go


In [94]:
from map import Map

In [95]:
df = pd.read_csv(r"C:\Users\au646858\OneDrive - Aarhus universitet\Documents\anemone_projects\es5\site.csv")

lon_span = np.linspace(df["longitude"].min(), df["longitude"].max(), 100)
lat_span = np.linspace(df["latitude"].min(), df["latitude"].max(), 100)

x = lon_span-lon_span.mean()
y = lat_span-lat_span.mean()

r2 = x[:, None]**2 + y[None, :]**2
r = np.sqrt(r2)

In [96]:
m1 = Map(data=df)

In [97]:
import plotly.graph_objects as go

m1 = Map(data=df)
m1.settings.markersize=0.
fig = m1.render()

# Option 1: Density heatmap from point data
fig.add_trace(go.Densitymap(
    lat=df["latitude"],
    lon=df["longitude"],
    z=df["doi"],   # values to color by
    radius=10,              # pixel radius of influence
    colorscale="Viridis",
    opacity=1.,
    showscale=True,
))
fig

In [7]:
import plotly.graph_objects as go
import json

# Build GeoJSON grid of rectangles
features = []
z_values = []
ids = []

dlon = (lon_span[1] - lon_span[0]) / 2
dlat = (lat_span[1] - lat_span[0]) / 2

for i, lat in enumerate(lat_span):
    for j, lon in enumerate(lon_span):
        cell_id = f"{i}-{j}"
        ids.append(cell_id)
        z_values.append(float(r[i, j]))
        features.append({
            "type": "Feature",
            "id": cell_id,
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [lon - dlon, lat - dlat],
                    [lon + dlon, lat - dlat],
                    [lon + dlon, lat + dlat],
                    [lon - dlon, lat + dlat],
                    [lon - dlon, lat - dlat],
                ]]
            }
        })

geojson = {"type": "FeatureCollection", "features": features}

In [89]:
def add_scalebar(fig, lat, lon, distance_km=10, n_segments=4, bar_width=6, height_factor=0.15):
    km_per_deg_lon = 111.32 * np.cos(np.radians(lat))
    total_dlon = distance_km / km_per_deg_lon
    seg_dlon = total_dlon / n_segments
    seg_km = distance_km / n_segments
    dlat = total_dlon * height_factor

    colors = ["black", "white"]
    group = "scalebar"
    first = True

    def _add(trace):
        nonlocal first
        trace.legendgroup = group
        if first:
            trace.showlegend = True
            trace.name = "Scale"
            first = False
        else:
            trace.showlegend = False
        fig.add_trace(trace)

    for i in range(n_segments):
        seg_lon0 = lon + i * seg_dlon
        seg_lon1 = lon + (i + 1) * seg_dlon
        _add(go.Scattermap(
            lat=[lat, lat],
            lon=[seg_lon0, seg_lon1],
            mode="lines",
            line=dict(width=bar_width, color=colors[i % 2]),
            hoverinfo="skip",
        ))

    for dlat_offset in [-bar_width * 0.0001, bar_width * 0.0001]:
        _add(go.Scattermap(
            lat=[lat + dlat_offset, lat + dlat_offset],
            lon=[lon, lon + total_dlon],
            mode="lines",
            line=dict(width=1, color="black"),
            hoverinfo="skip",
        ))

    for i in range(n_segments + 1):
        tick_lon = lon + i * seg_dlon
        _add(go.Scattermap(
            lat=[lat - dlat, lat + dlat],
            lon=[tick_lon, tick_lon],
            mode="lines",
            line=dict(width=2, color="black"),
            hoverinfo="skip",
        ))

    for i in [0, n_segments]:
        label_lon = lon + i * seg_dlon
        label_km = int(i * seg_km)
        _add(go.Scattermap(
            lat=[lat - dlat * 2.5],
            lon=[label_lon],
            mode="text",
            text=[f"{label_km}"],
            textfont=dict(size=12, color="black"),
            hoverinfo="skip",
        ))

    _add(go.Scattermap(
        lat=[lat - dlat * 2.5],
        lon=[lon + total_dlon + seg_dlon * 0.4],
        mode="text",
        text=["km"],
        textfont=dict(size=12, color="black"),
        hoverinfo="skip",
    ))

In [ ]:
# def add_scalebar(fig, x=0.05, y=0.05, width=0.2, height=0.012,
#                        distance_km=10, n_segments=4):
#     """Add a fixed scalebar in paper coordinates (fraction of figure).

#     Args:
#         fig: Plotly figure
#         x, y: Bottom-left anchor in paper coords (0–1)
#         width: Total bar width as fraction of figure
#         height: Bar height as fraction of figure
#         distance_km: Total distance label
#         n_segments: Number of alternating segments
#     """
#     seg_width = width / n_segments
#     seg_km = distance_km / n_segments
#     colors = ["black", "white"]

#     shapes = list(fig.layout.shapes or [])
#     annotations = list(fig.layout.annotations or [])

#     # Alternating segments
#     for i in range(n_segments):
#         shapes.append(dict(
#             type="rect",
#             xref="paper", yref="paper",
#             x0=x + i * seg_width,
#             y0=y,
#             x1=x + (i + 1) * seg_width,
#             y1=y + height,
#             fillcolor=colors[i % 2],
#             line=dict(color="black", width=1),
#         ))

#     # Tick labels at start and end
#     for i in [0, n_segments]:
#         annotations.append(dict(
#             x=x + i * seg_width,
#             y=y - 0.005,
#             xref="paper", yref="paper",
#             text=str(int(i * seg_km)),
#             showarrow=False,
#             font=dict(size=11, color="black"),
#             yanchor="top",
#             xanchor="center",
#         ))

#     # "km" label
#     annotations.append(dict(
#         x=x + width + 0.015,
#         y=y + height / 2,
#         xref="paper", yref="paper",
#         text="km",
#         showarrow=False,
#         font=dict(size=11, color="black"),
#         yanchor="middle",
#         xanchor="left",
#     ))

#     fig.update_layout(shapes=shapes, annotations=annotations)

In [ ]:
fig = m1.render()
m1.settings.markersize=3.
fig.add_trace(go.Choroplethmap(
    geojson=geojson,
    locations=ids,
    z=z_values,
    colorscale="Turbo",
    marker=dict(opacity=1, line=dict(width=0)),
    showscale=True,
    hoverinfo="skip",       # no hover tooltip
    selectedpoints=[],       # nothing pre-selected
    unselected=dict(marker=dict(opacity=1)),  # keep appearance on selection
))
# fig.update_layout(
#     clickmode="event",       # don't select on click
# )
data_list = list(fig.data)
fig.data = [data_list[-1]] + data_list[:-1]

bar_lat = df["latitude"].min() - 0.005
bar_lon = df["longitude"].min()

add_scalebar(fig, distance_km=10, n_segments=4)#, height_factor=0.01)
# fig.update_layout(showlegend=True,
#     legend=dict(
#         x=0.01,
#         y=0.99,
#         xanchor="left",
#         yanchor="top",
#         bgcolor="rgba(255,255,255,0.5)",  # semi-transparent background
#         bordercolor="black",
#         borderwidth=1,
#     )
# )
fig

In [60]:
from scipy.spatial import cKDTree

points = np.column_stack([df["longitude"].values, df["latitude"].values])
tree = cKDTree(points)

# Hex radius from median nearest-neighbor distance
dists, _ = tree.query(points, k=2)
default_radius = np.median(dists[:, 1]) * 3

margin = 0.01
bbox = [
    points[:, 0].min() - margin, points[:, 0].max() + margin,
    points[:, 1].min() - margin, points[:, 1].max() + margin,
]

def make_hexagon(cx, cy, radius):
    angles = np.linspace(0, 2 * np.pi, 25)[:-1]
    return [[cx + radius * np.cos(a), cy + radius * np.sin(a)] for a in angles]

def clip_to_bisector(poly, center, neighbor):
    """Clip polygon to the half-plane closer to center than neighbor."""
    mid = [(center[0] + neighbor[0]) / 2, (center[1] + neighbor[1]) / 2]
    nx = center[0] - neighbor[0]
    ny = center[1] - neighbor[1]

    def inside(p):
        return (p[0] - mid[0]) * nx + (p[1] - mid[1]) * ny >= 0

    def intersect(p1, p2):
        d1 = (p1[0] - mid[0]) * nx + (p1[1] - mid[1]) * ny
        d2 = (p2[0] - mid[0]) * nx + (p2[1] - mid[1]) * ny
        t = d1 / (d1 - d2)
        return [p1[0] + t * (p2[0] - p1[0]), p1[1] + t * (p2[1] - p1[1])]

    clipped = []
    prev = poly[-1]
    for curr in poly:
        if inside(curr):
            if not inside(prev):
                clipped.append(intersect(prev, curr))
            clipped.append(curr)
        elif inside(prev):
            clipped.append(intersect(prev, curr))
        prev = curr
    return clipped

def clip_to_bbox(poly, bbox):
    xmin, xmax, ymin, ymax = bbox
    def clip_edge(poly, inside_fn, intersect_fn):
        if not poly:
            return []
        clipped = []
        prev = poly[-1]
        for curr in poly:
            if inside_fn(curr):
                if not inside_fn(prev):
                    clipped.append(intersect_fn(prev, curr))
                clipped.append(curr)
            elif inside_fn(prev):
                clipped.append(intersect_fn(prev, curr))
            prev = curr
        return clipped
    poly = clip_edge(poly, lambda p: p[0] >= xmin,
        lambda p1, p2: [xmin, p1[1] + (p2[1]-p1[1])*(xmin-p1[0])/(p2[0]-p1[0])])
    poly = clip_edge(poly, lambda p: p[0] <= xmax,
        lambda p1, p2: [xmax, p1[1] + (p2[1]-p1[1])*(xmax-p1[0])/(p2[0]-p1[0])])
    poly = clip_edge(poly, lambda p: p[1] >= ymin,
        lambda p1, p2: [p1[0] + (p2[0]-p1[0])*(ymin-p1[1])/(p2[1]-p1[1]), ymin])
    poly = clip_edge(poly, lambda p: p[1] <= ymax,
        lambda p1, p2: [p1[0] + (p2[0]-p1[0])*(ymax-p1[1])/(p2[1]-p1[1]), ymax])
    return poly

# Find neighbors within reach
neighbor_indices = tree.query_ball_point(points, r=default_radius * 3)

features = []
ids = []
z_values = []

for idx in range(len(points)):
    hex_verts = make_hexagon(points[idx, 0], points[idx, 1], default_radius)

    # Clip against perpendicular bisector with each neighbor
    for nb_idx in neighbor_indices[idx]:
        if nb_idx == idx:
            continue
        hex_verts = clip_to_bisector(hex_verts, points[idx], points[nb_idx])
        if len(hex_verts) < 3:
            break

    hex_verts = clip_to_bbox(hex_verts, bbox)

    if len(hex_verts) < 3:
        continue

    coords = [[v[0], v[1]] for v in hex_verts]
    coords.append(coords[0])

    cell_id = str(idx)
    ids.append(cell_id)
    z_values.append(float(df["doi"].iloc[idx]))
    features.append({
        "type": "Feature",
        "id": cell_id,
        "geometry": {"type": "Polygon", "coordinates": [coords]}
    })

geojson = {"type": "FeatureCollection", "features": features}

In [81]:
import geopandas as gpd
from shapely.geometry import MultiPoint, Point, Polygon
from shapely.ops import unary_union
from scipy.spatial import cKDTree

points_xy = np.column_stack([df["longitude"].values, df["latitude"].values])
z_data = df["doi"].values

# --- 1. Buffer around all data points ---
tree = cKDTree(points_xy)
nn_dists, _ = tree.query(points_xy, k=2)
cell_size = np.median(nn_dists[:, 1])
buffer_dist = cell_size * 2

mp = MultiPoint([Point(p) for p in points_xy])
buffered = mp.buffer(buffer_dist, resolution=16)

# --- 2. Generate hexagonal mesh inside the buffer ---
xmin, ymin, xmax, ymax = buffered.bounds
dx = cell_size
dy = cell_size * np.sqrt(3) / 2

mesh_points = []
row = 0
y = ymin
while y <= ymax:
    x = xmin + (dx / 2 if row % 2 else 0)
    while x <= xmax:
        p = Point(x, y)
        if buffered.contains(p):
            mesh_points.append([x, y])
        x += dx
    y += dy
    row += 1

mesh_points = np.array(mesh_points)
print(f"Generated {len(mesh_points)} mesh points inside buffer")

# --- 3. IDW interpolation ---
def idw_interpolate(mesh_pts, data_pts, data_vals, power=2, k=12):
    tree = cKDTree(data_pts)
    dists, idxs = tree.query(mesh_pts, k=min(k, len(data_pts)))
    dists = np.maximum(dists, 1e-12)
    weights = 1.0 / dists**power
    weighted_vals = np.sum(weights * data_vals[idxs], axis=1)
    return weighted_vals / np.sum(weights, axis=1)

z_mesh = idw_interpolate(mesh_points, points_xy, z_data, k=7)

# --- 4. Build hex cells with bisector clipping ---
# Start with oversized hex — bisectors will trim to correct Voronoi-like tiling
hex_radius = cell_size * 1.2
hex_angles = np.linspace(0, 2 * np.pi, 7)[:-1]

def clip_to_bisector(poly_coords, center, neighbor):
    mid = [(center[0] + neighbor[0]) / 2, (center[1] + neighbor[1]) / 2]
    nx = center[0] - neighbor[0]
    ny = center[1] - neighbor[1]

    def inside(p):
        return (p[0] - mid[0]) * nx + (p[1] - mid[1]) * ny >= 0

    def intersect(p1, p2):
        d1 = (p1[0] - mid[0]) * nx + (p1[1] - mid[1]) * ny
        d2 = (p2[0] - mid[0]) * nx + (p2[1] - mid[1]) * ny
        t = d1 / (d1 - d2)
        return [p1[0] + t * (p2[0] - p1[0]), p1[1] + t * (p2[1] - p1[1])]

    clipped = []
    prev = poly_coords[-1]
    for curr in poly_coords:
        if inside(curr):
            if not inside(prev):
                clipped.append(intersect(prev, curr))
            clipped.append(curr)
        elif inside(prev):
            clipped.append(intersect(prev, curr))
        prev = curr
    return clipped

# Search all neighbors within reach of the oversized hex
mesh_tree = cKDTree(mesh_points)
neighbor_radius = hex_radius * 2.5
mesh_neighbors = mesh_tree.query_ball_point(mesh_points, r=neighbor_radius)

features = []
ids = []
z_values = []

for i, (cx, cy) in enumerate(mesh_points):
    coords = [[cx + hex_radius * np.cos(a), cy + hex_radius * np.sin(a)]
              for a in hex_angles]

    for nb_idx in mesh_neighbors[i]:
        if nb_idx == i:
            continue
        coords = clip_to_bisector(coords, [cx, cy], list(mesh_points[nb_idx]))
        if len(coords) < 3:
            break

    # if len(coords) < 3:
    #     continue

    # cell_poly = Polygon(coords)
    # clipped = cell_poly.intersection(buffered)

    # if clipped.is_empty or clipped.area < 1e-15:
    #     continue
    # if clipped.geom_type == 'MultiPolygon':
    #     clipped = max(clipped.geoms, key=lambda g: g.area)
    # if clipped.geom_type != 'Polygon':
    #     continue

    # ext_coords = list(clipped.exterior.coords)
    ext_coords = coords

    cell_id = str(i)
    ids.append(cell_id)
    z_values.append(float(z_mesh[i]))
    features.append({
        "type": "Feature",
        "id": cell_id,
        "geometry": {
            "type": "Polygon",
            "coordinates": [[[c[0], c[1]] for c in ext_coords]]
        }
    })

geojson = {"type": "FeatureCollection", "features": features}
print(f"Built {len(features)} hex cells")

Generated 304 mesh points inside buffer
Built 304 hex cells


In [92]:
fig = m1.render()
m1.settings.markersize=3.
fig.add_trace(go.Choroplethmap(
    geojson=geojson,
    locations=ids,
    z=z_values,
    colorscale="Turbo",
    marker=dict(opacity=1, line=dict(width=0, color=None)),
    showscale=False,
    hoverinfo="skip",
    selectedpoints=[],
    unselected=dict(marker=dict(opacity=1)),
    name="Grid",   # legend label
    showlegend=True              # ensure it appears in the legend
))
# fig.update_layout(
#     clickmode="event",       # don't select on click
# )
data_list = list(fig.data)
fig.data = [data_list[-1]] + data_list[:-1]
bar_lat = df["latitude"].min() - 0.005
bar_lon = df["longitude"].min()

add_scalebar(fig, bar_lat, bar_lon, distance_km=3, n_segments=3, height_factor=0.01)
fig.update_layout(showlegend=True,
    legend=dict(
        x=0.01,
        y=0.99,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.5)",  # semi-transparent background
        bordercolor="black",
        borderwidth=1,
    )
)
fig